# Conveyor perception

**An end-to-end industrial CV pipeline on a free T4:** real recycling data → trained model → live detection → drift monitoring → triage decisions.

- **Runtime:** Google Colab T4 (free tier, ~12h cap).  
- **Data:** bundled 4-class recycling set (CC BY 4.0).  
- **Model:** YOLO26s, trained in-kernel, cached on re-run.  
- **Goal:** show the loop — train → infer → drift → triage → maintain — on real data, in <5 minutes.


In [ ]:
# --- Cell 1: Runtime + env check ---
import os, sys, json, platform
from pathlib import Path

# --- 1. Colab or local? ---
IN_COLAB = 'google.colab' in sys.modules
print(f'  Runtime: {"Google Colab" if IN_COLAB else "Local (" + platform.node() + ")"}')

# --- 2. Python + key libs (skip import if missing) ---
print(f'  Python: {sys.version.split()[0]}  ({sys.executable.split("/")[-1]})')
for mod in ['numpy', 'torch', 'ultralytics', 'supervision', 'roboflow']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'  {mod:14s} {v}')
    except ImportError:
        print(f'  {mod:14s} — not installed yet (cell 2 will install)')

# --- 3. GPU (or warn if CPU-only) ---
_gpu = 'unknown'
try:
    import torch
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except Exception:
    pass
print(f'  GPU:    {_gpu}')

# --- 4. Disk + RAM ---
_disk_free = '?'
try:
    import shutil
    _u = shutil.disk_usage('/')
    _disk_free = f'{_u.free / 1e9:.1f} GB free of {_u.total / 1e9:.1f} GB'
except Exception:
    pass
print(f'  Disk:   {_disk_free}')

# --- 5. Locate the repo (for local runs) — Colab gets cloned by cell 2 ---
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
if not IN_COLAB:
    # Walk up until we find the repo root (contains pyproject.toml)
    while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
        REPO = REPO.parent
print(f'  Repo:   {REPO}{" (will be cloned here by cell 2)" if IN_COLAB else ""}')

# NOTE: colab_session + the state singleton are intentionally NOT used here.
# Cell 1 runs BEFORE the clone (cell 2), so the repo isn't on disk yet — any
# `import colab_session` would crash with ModuleNotFoundError. State init is
# deferred to cell 3, which runs after the clone + install are done. (Aug 22 2026)
print()
print('  ✓ env check done.  Next: cell 2 (clone + install).')


In [ ]:
# --- Cell 2: Install + clone ---
import os, sys, subprocess, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Clone repo (idempotent — skip if pyproject.toml already there) ---
if IN_COLAB:
    if (REPO / 'pyproject.toml').exists():
        print(f'  Repo already at {REPO} (skipping clone)')
    else:
        print(f'  Cloning conveyor-perception -> {REPO}...')
        REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/roniejosephv-star/conveyor-perception.git',
             str(REPO)],
            check=True,
        )
        print('  ✓ cloned.')
else:
    print(f'  Local repo at {REPO} (skipping clone)')

# --- 2. Add REPO + REPO/notebooks to sys.path (colab_session.py lives in notebooks/) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)
print(f'  Path:  {REPO}  +  {REPO}/notebooks')

# --- 3. Install minimal open-source deps (numpy/torch already on Colab) ---
INSTALL = ['ultralytics', 'supervision', 'opencv-python-headless', 'roboflow']
print(f'  Installing: {", ".join(INSTALL)}')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', *INSTALL]
)
print('  ✓ installed.')

# --- 4. Verify the key imports work (smoke test) ---
print('  Verifying:')
for _mod in ['ultralytics', 'supervision', 'roboflow']:
    try:
        _m = importlib.import_module(_mod)
        _v = getattr(_m, '__version__', '?')
        print(f'    {_mod:14s} {_v}  ok')
    except ImportError as _e:
        print(f'    {_mod:14s} FAIL  {_e}')

# --- 5. Verify colab_session is now importable (the import cell 3 needs) ---
try:
    from colab_session import get_state
    print('    colab_session  ok  (get_state() ready for cell 3)')
except ImportError as _e:
    print(f'    colab_session  FAIL  {_e}')

# --- 6. Quick sanity check: list the repo top-level ---
print(f'  Repo top-level:')
for _entry in sorted(REPO.iterdir()):
    if not _entry.name.startswith('.'):
        print(f'    {_entry.name}')

print()
print('  ✓ install + clone done.  Next: cell 3 (state + toggles).')


In [ ]:
# --- Cell 3: State + toggles ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Defensive install: ipywidgets is needed for the toggle UI ---
# (Colab has it pre-installed, but cell 3 should not assume cell 2 installed it.)
try:
    import ipywidgets  # noqa: F401
    print('  ipywidgets: ok')
except ImportError:
    print('  Installing ipywidgets (toggle UI dep)...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'ipywidgets']
    )
    print('  ✓ installed.')

# --- 2. Idempotent sys.path setup (cell 2 already did this; redo is harmless) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 3. Create the SessionState singleton (lives in builtins so every cell sees it) ---
from colab_session import get_state, toggle_ui

state = get_state()
state.log('cell-3', action='init', note='state singleton + toggle UI ready')

# --- 4. Show the toggle UI (4 framework abstractions + 8 JD modules) ---
# Untick anything you want the pipeline to skip. Defaults: all enabled.
print()
print('  Toggle the components below — defaults are all enabled.')
print('  Untick anything you want to skip in the pipeline.')
print()
ui = toggle_ui()
display(ui)

# --- 5. Print a summary of what's currently enabled (grouped) ---
print()
print('  Current toggles:')
abstr_keys = [k for k in state.toggles if k.startswith('abstraction:')]
mod_keys = [k for k in state.toggles if k.startswith('module:')]
print('    4 framework abstractions:')
for _k in abstr_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')
print('    8 JD modules:')
for _k in mod_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')

_n_on = sum(state.toggles.values())
_n_total = len(state.toggles)
print()
print(f'  ✓ state + toggles ready.  {_n_on}/{_n_total} components enabled.')
print('  Next: cell 4 (load abstractions).')


In [ ]:
# --- Cell 4: Load abstractions ---
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup: src/ (for `import conveyor_perception`) + REPO/notebooks ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Import the 4 abstraction classes (unconditional — fail fast on missing module) ---
from colab_session import get_state
from conveyor_perception.core.detection_pipeline import DetectionPipeline as Detector
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.triage_surface import MCPTriageSurface, InMemoryAlertQueue

state = get_state()
loaded: dict = {}
skipped: list = []

# --- 3. Toggle-gated load (the toggle UI from cell 3 controls this) ---
if state.toggles.get('abstraction:detector', True):
    # Detector is just a class ref for now — model is wired in cell 8 after training.
    loaded['detector'] = Detector
    print('  ✓ Detector class loaded (YOLO26 + OpenCV DNN)')
else:
    skipped.append('abstraction:detector')
    print('  ○ Detector skipped (toggle off)')

if state.toggles.get('abstraction:tracker', True):
    loaded['tracker'] = TrackingPipeline()
    print('  ✓ TrackingPipeline instantiated (ByteTrack)')
else:
    skipped.append('abstraction:tracker')
    print('  ○ TrackingPipeline skipped (toggle off)')

if state.toggles.get('abstraction:drift_monitor', True):
    loaded['drift_monitor'] = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
    print('  ✓ DriftMonitor instantiated (KS test + z-score + MAD)')
else:
    skipped.append('abstraction:drift_monitor')
    print('  ○ DriftMonitor skipped (toggle off)')

if state.toggles.get('abstraction:triage', True):
    loaded['triage_surface'] = MCPTriageSurface('l1-triage', InMemoryAlertQueue())
    print('  ✓ MCPTriageSurface instantiated (5 MCP tools)')
else:
    skipped.append('abstraction:triage')
    print('  ○ MCPTriageSurface skipped (toggle off)')

state.log('cell-4', action='load-abstractions', loaded=list(loaded.keys()), skipped=skipped)
print()
print(f'  ✓ loaded {len(loaded)}/4 abstractions, skipped {len(skipped)}.')
print('  Next: cell 5 (load modules).')


In [ ]:
# --- Cell 5: Load modules ---
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup (idempotent — cell 4 already added REPO/src) ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Module metadata: toggle key + import path + 1-line description ---
from colab_session import get_state
state = get_state()

MODULES_META = [
    ('module:perception',             'conveyor_perception.perception',             'UltralyticsDetector + RecyclingInferenceService'),
    ('module:triage',                 'conveyor_perception.triage',                 'L1TriageAgent + 7 severity rules'),
    ('module:predictive_maintenance', 'conveyor_perception.predictive_maintenance', 'MaintenanceAdvisor + 3 signal types'),
    ('module:multitask',              'conveyor_perception.multitask',              'MultitaskPipeline (Detector->Tracker->Drift->Triage)'),
    ('module:integration',            'conveyor_perception.integration',            'ConveyorNode (real ROS 2) + MockROS2Node (CI)'),
    ('module:robustness',             'conveyor_perception.robustness',             'RobustnessTestSuite + 13 augmentations'),
    ('module:monitoring',             'conveyor_perception.monitoring',             'MonitoringDashboard + ShiftReport'),
    ('module:optimization',           'conveyor_perception.optimization',           'benchmark_pytorch/onnx + export_onnx'),
]

loaded: list = []
skipped: list = []
failed: list = []

# --- 3. Toggle-gated dynamic load (one bad module doesn't kill the cell) ---
print('  Loading 8 JD modules:')
for _toggle_key, _module_path, _desc in MODULES_META:
    if not state.toggles.get(_toggle_key, True):
        skipped.append(_toggle_key)
        print(f'    ○ {_module_path}  (disabled by toggle)')
        continue
    try:
        importlib.import_module(_module_path)
        loaded.append(_module_path)
        print(f'    ✓ {_module_path}')
        print(f'        {_desc}')
    except Exception as _e:
        failed.append((_module_path, str(_e)))
        print(f'    ✗ {_module_path}  failed: {_e}')

state.log(
    'cell-5',
    action='load-modules',
    loaded=loaded,
    skipped=skipped,
    failed=[m for m, _ in failed],
)
print()
print(f'  ✓ loaded {len(loaded)}/{len(MODULES_META)} modules, skipped {len(skipped)}, failed {len(failed)}.')
print('  Next: cell 6 (data registry).')


In [ ]:
# --- Cell 6: Data registry ---
import os, sys, subprocess, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Defensive PyYAML install (data.yaml is YAML) ---
try:
    import yaml  # noqa: F401
except ImportError:
    print('  Installing PyYAML (data.yaml parser)...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'pyyaml']
    )
    print('  ✓ installed.')
    import yaml

# --- 2. Path setup (idempotent) ---
SRC = REPO / 'src'
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 3. Scan data/ for datasets with data.yaml ---
from colab_session import get_state
state = get_state()

DATA_ROOTS = [REPO / 'data' / 'sample', REPO / 'data' / 'raw']

def _count_imgs(_d: Path, _rel_path: str) -> int:
    if not _rel_path:
        return 0
    _p = _d / _rel_path.lstrip('./').lstrip('/')
    if not _p.exists():
        return 0
    return len(list(_p.glob('*.jpg'))) + len(list(_p.glob('*.png')))

def _dir_size_mb(_d: Path) -> float:
    if not _d.exists():
        return 0.0
    return sum(f.stat().st_size for f in _d.rglob('*') if f.is_file()) / (1024 * 1024)

registry: list = []
for _root in DATA_ROOTS:
    if not _root.exists():
        continue
    for _d in sorted(_root.iterdir()):
        if not _d.is_dir():
            continue
        _yaml_p = _d / 'data.yaml'
        if not _yaml_p.exists():
            continue
        try:
            _cfg = yaml.safe_load(_yaml_p.read_text())
        except Exception as _e:
            print(f'  ✗ could not parse {_yaml_p}: {_e}')
            _cfg = {}
        _nc = _cfg.get('nc', len(_cfg.get('names', [])))
        _names = _cfg.get('names', [])
        _n_train = _count_imgs(_d, _cfg.get('train', 'train/images'))
        _n_val = _count_imgs(_d, _cfg.get('val', 'val/images'))
        _meta_p = _d / 'dataset_meta.json'
        _meta = json.loads(_meta_p.read_text()) if _meta_p.exists() else {}
        _status = 'bundled' if _root == REPO / 'data' / 'sample' else 'downloaded'
        _size_mb = _dir_size_mb(_d)
        registry.append({
            'name': _d.name,
            'path': str(_d),
            'status': _status,
            'n_train': _n_train,
            'n_val': _n_val,
            'nc': _nc,
            'names': _names,
            'source': _meta.get('source', '—'),
            'license': _meta.get('license', '—'),
            'baseline_mAP50': _meta.get('pre_trained_baseline_mAP50'),
            'size_mb': _size_mb,
        })

# --- 4. Cache the registry to state (downstream cells read it) ---
state.dataset_registry = registry
state.metric('datasets_available', len(registry))
state.log(
    'cell-6',
    action='data-registry',
    n_datasets=len(registry),
    names=[_r['name'] for _r in registry],
)

# --- 5. Render as a text table ---
W = 82
print()
print('─' * W)
print(f'  DATA REGISTRY  ({len(registry)} dataset(s) on disk)'.center(W))
print('─' * W)
if not registry:
    print('  (no datasets found — run cell 7 to download the recycling_v3 set)')
else:
    for _r in registry:
        print(f'  • {_r["name"]}  [{_r["status"]}]')
        print(f'      path:    {_r["path"]}')
        print(f'      images:  {_r["n_train"]} train  /  {_r["n_val"]} val  /  {_r["size_mb"]:.1f} MB')
        print(f'      classes ({_r["nc"]}): {
.join(str(_n) for _n in _r["names"])}')
        if _r.get('source') and _r['source'] != '—':
            print(f'      source:  {_r["source"]}')
        if _r.get('license') and _r['license'] != '—':
            print(f'      license: {_r["license"]}')
        if _r.get('baseline_mAP50') is not None:
            print(f'      baseline mAP50: {_r["baseline_mAP50"]}')
        print()
print('─' * W)
if registry:
    print(f'  ✓ {len(registry)} dataset(s) registered.  Next: cell 7 (download if you need a bigger one).')
else:
    print('  ✓ registry ready (empty).  Next: cell 7 (download).')


In [ ]:
# --- Cell 7: Data download (idempotent) ---
import os, sys, shutil
from pathlib import Path
from colab_session import get_state

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
DATA_RAW = REPO / 'data' / 'raw'

# Change TARGET_NAME to download a different dataset (or 'skip' to do nothing).
TARGET_NAME = 'recycling_v3'  # 'recycling_v3' | 'everyday_recycle_waste' | 'recycling_classification' | 'skip'

DATASETS = {
    'recycling_v3':             dict(workspace='zkf624',                       project='-recycling',               version=3),
    'everyday_recycle_waste':   dict(workspace='everyday-recycle-waste-xhayk', project='everyday-recycle-waste',   version=3),
    'recycling_classification': dict(workspace='new-workspace-eosax',          project='recycling-classification', version=1),
}

SAFE_NAME = TARGET_NAME.replace('-', '_')
TARGET = DATA_RAW / SAFE_NAME
DATA_YAML = TARGET / 'data.yaml'

def _count_imgs(_p: Path) -> int:
    if not _p.exists():
        return 0
    return len(list(_p.glob('*.jpg'))) + len(list(_p.glob('*.png')))

def _dir_size_mb(_d: Path) -> float:
    if not _d.exists():
        return 0.0
    return sum(f.stat().st_size for f in _d.rglob('*') if f.is_file()) / (1024 * 1024)

W = 82
print()
print('─' * W)
print(f'  DOWNLOAD CHECK  —  target: {SAFE_NAME}'.center(W))
print('─' * W)

if DATA_YAML.exists():
    # 1. Already on disk — skip the download (idempotent re-run path).
    n_train = _count_imgs(TARGET / 'train' / 'images')
    n_val = _count_imgs(TARGET / 'val' / 'images')
    n_test = _count_imgs(TARGET / 'test' / 'images')
    size_mb = _dir_size_mb(TARGET)
    print(f'  ✓ {SAFE_NAME} already on disk — download skipped')
    print()
    print(f'    train:  {n_train:>6} images')
    print(f'    val:    {n_val:>6} images' + (f'   (incl. test: {n_test})' if n_test else ''))
    print(f'    size:   {size_mb:>7.1f} MB on disk')
    print(f'    path:   {TARGET}')
    print('─' * W)
    state.metric(f'dataset_{SAFE_NAME}_status', 'already_present')
    state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
    state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
    state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
    state.log(
        'cell-7',
        action='skip',
        reason='already-present',
        dataset=SAFE_NAME,
        n_train=n_train,
        n_val=n_val,
        size_mb=round(size_mb, 1),
    )
elif TARGET_NAME == 'skip':
    print('  TARGET_NAME=skip — no download attempted.')
    print('─' * W)
    state.metric('dataset_download_status', 'skipped')
else:
    # 2. Try Roboflow Universe (public demo key, read-only for CC-BY 4.0 datasets).
    cfg = DATASETS.get(SAFE_NAME)
    if cfg is None:
        print(f'  ✗ Unknown dataset: {TARGET_NAME!r}')
        print(f'  Add an entry to DATASETS in this cell, then re-run.')
        print('─' * W)
        state.metric(f'dataset_{SAFE_NAME}_status', 'unknown')
    else:
        print(f'  Downloading {SAFE_NAME} from Roboflow Universe...')
        print(f'    workspace: {cfg["workspace"]}')
        print(f'    project:   {cfg["project"]}')
        print(f'    version:   {cfg["version"]}')
        print('─' * W)
        _ok = False
        _err = None
        try:
            from roboflow import Roboflow
            _api_key = 'qogO5hAuLgUUYMbNT6W3'  # public demo key, read-only
            _env_p = REPO / '.env'
            if _env_p.exists():
                for _line in _env_p.read_text().splitlines():
                    if _line.startswith('ROBOFLOW_API_KEY='):
                        _api_key = _line.split('=', 1)[1].strip() or _api_key
            _rf = Roboflow(api_key=_api_key)
            _project = _rf.workspace(cfg['workspace']).project(cfg['project'])
            _version = _project.version(cfg['version'])
            # 'yolov11' is the Roboflow LABEL format (YOLO .txt annotations
            # + folder layout), NOT the model architecture. The YOLO .txt schema
            # is identical across YOLOv5/8/11/26 — the actual model choice
            # (yolo26s) happens in cell 8 via YOLO('yolo26s.pt').
            _dataset = _version.download('yolov11')
            _downloaded = Path(_dataset.location)
            if _downloaded.exists():
                DATA_RAW.mkdir(parents=True, exist_ok=True)
                if TARGET.exists():
                    shutil.rmtree(TARGET)
                _downloaded.rename(TARGET)
                _ok = True
        except ImportError as _e:
            _err = f'roboflow not installed ({_e})'
        except Exception as _e:
            _err = f'{type(_e).__name__}: {_e}'

        if _ok:
            n_train = _count_imgs(TARGET / 'train' / 'images')
            n_val = _count_imgs(TARGET / 'val' / 'images')
            size_mb = _dir_size_mb(TARGET)
            print(f'  ✓ Downloaded {SAFE_NAME}')
            print()
            print(f'    train:  {n_train:>6} images')
            print(f'    val:    {n_val:>6} images')
            print(f'    size:   {size_mb:>7.1f} MB on disk')
            print(f'    path:   {TARGET}')
            print('─' * W)
            state.metric(f'dataset_{SAFE_NAME}_status', 'downloaded')
            state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
            state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
            state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
            state.log(
                'cell-7',
                action='downloaded',
                dataset=SAFE_NAME,
                n_train=n_train,
                n_val=n_val,
                size_mb=round(size_mb, 1),
            )
        else:
            print(f'  ✗ Download failed: {_err}')
            print()
            print('  Manual fallback:')
            print(f'    1. Open https://universe.roboflow.com/{cfg["workspace"]}/{cfg["project"]}/dataset/{cfg["version"]}')
            print('    2. Click Download Dataset -> Format: YOLOv11')
            print(f'    3. Extract the zip into {DATA_RAW}/ so the path is:')
            print(f'       {DATA_RAW}/{SAFE_NAME}/data.yaml')
            print('─' * W)
            state.metric(f'dataset_{SAFE_NAME}_status', 'failed')
            state.log('cell-7', action='download-failed', error=str(_err), dataset=SAFE_NAME)

# 3. Refresh the registry on state (so cell 8 sees the new dataset).
import yaml as _yaml
_registry = []
for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
    if not _root.exists():
        continue
    for _d in sorted(_root.iterdir()):
        if not _d.is_dir():
            continue
        _yp = _d / 'data.yaml'
        if not _yp.exists():
            continue
        try:
            _cfg = _yaml.safe_load(_yp.read_text())
        except Exception:
            _cfg = {}
        _nc = _cfg.get('nc', len(_cfg.get('names', [])))
        _names = _cfg.get('names', [])
        _n_train = _count_imgs(_d / _cfg.get('train', 'train/images').lstrip('./').lstrip('/'))
        _n_val = _count_imgs(_d / _cfg.get('val', 'val/images').lstrip('./').lstrip('/'))
        _status = 'bundled' if _root == REPO / 'data' / 'sample' else 'downloaded'
        _size_mb = _dir_size_mb(_d)
        _registry.append({
            'name': _d.name,
            'path': str(_d),
            'status': _status,
            'n_train': _n_train,
            'n_val': _n_val,
            'nc': _nc,
            'names': _names,
            'size_mb': _size_mb,
        })
state.dataset_registry = _registry
state.metric('datasets_available', len(_registry))
print()
print(f'  ✓ {len(_registry)} dataset(s) now in registry.  Next: cell 8 (train).')


In [ ]:
# --- Cell 8: Train (cached on re-run) ---
import os, sys, time, csv
from pathlib import Path
from colab_session import get_state, pick_device

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
state = get_state()

# --- 0. Pick the dataset (change DATASET_NAME to switch) ---
DATASET_NAME = 'recycling_demo'  # 'recycling_demo' (bundled, ~1 min) or 'recycling_v3' (downloaded, ~5-10 min)

# --- 1. Resolve the dataset path from the registry (or fall back to disk scan) ---
_reg = getattr(state, 'dataset_registry', None) or []
_match = next((_r for _r in _reg if _r['name'] == DATASET_NAME), None)
if _match is None:
    for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
        _cand = _root / DATASET_NAME
        if (_cand / 'data.yaml').exists():
            _match = {'name': DATASET_NAME, 'path': str(_cand), 'n_train': 0, 'n_val': 0, 'nc': 0, 'names': []}
            break

# --- 2. Compute the cache + toggle gate (skip = don't crash) ---
_skip_reason = None
if not state.toggles.get('module:perception', True):
    _skip_reason = 'toggle-off'
elif _match is None:
    _skip_reason = 'dataset-not-found'

if _skip_reason:
    print('─' * 72)
    if _skip_reason == 'toggle-off':
        print('  module:perception toggle is OFF — skipping training.')
        print('  Re-enable the toggle in cell 3 to train the model.')
    else:
        print(f'  ✗ Dataset {DATASET_NAME!r} not found in registry or on disk.')
        print('  Run cell 6 to scan, or cell 7 to download.')
    print('─' * 72)
    state.log('cell-8', action='skipped', reason=_skip_reason, dataset=DATASET_NAME)
    print('  Next: cell 9 (compare).')
else:
    DATA_DIR = Path(_match['path'])
    YAML_P = DATA_DIR / 'data.yaml'
    MODEL_DIR = REPO / 'models' / DATASET_NAME
    BEST_PT = MODEL_DIR / 'weights' / 'best.pt'
    RESULTS_CSV = MODEL_DIR / 'results.csv'

    print('─' * 72)
    _n_train = _match.get('n_train', 0) or 0
    _n_val = _match.get('n_val', 0) or 0
    _nc = _match.get('nc', 0) or 0
    print(f'  TRAIN  —  dataset: {DATASET_NAME}  (train: {_n_train}, val: {_n_val}, cls: {_nc})'.center(72))
    print('─' * 72)

    if BEST_PT.exists() and RESULTS_CSV.exists():
        # --- Cached path: read metrics from results.csv, no re-training ---
        print(f'  ✓ Cached model found at {BEST_PT}')
        print(f'      size: {BEST_PT.stat().st_size / 1e6:.1f} MB')
        try:
            with open(RESULTS_CSV) as _f:
                _rows = list(csv.DictReader(_f))
                _rows = [_r for _r in _rows if any(v.strip() for v in _r.values())]
            if _rows:
                _last = _rows[-1]
                _strip = lambda _k: _last.get(_k, '').strip()
                print()
                print('  -- Final epoch metrics (from results.csv) --')
                for _k in [
                    'epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
                    'metrics/precision(B)', 'metrics/recall(B)',
                    'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'lr/pg0',
                ]:
                    if _k in _last:
                        print(f'      {_k:30}  {_strip(_k)}')
                try:
                    _map50 = float(_strip('metrics/mAP50(B)'))
                    state.metric(f'map50_{DATASET_NAME}', _map50)
                    state.active_model_path = str(BEST_PT)
                    state.active_dataset = DATASET_NAME
                    state.log('cell-8', action='cached', dataset=DATASET_NAME, mAP50=_map50)
                except Exception:
                    pass
        except Exception as _e:
            print(f'  could not read results.csv: {_e}')
        print()
        print(f'  re-run skipped: cached model used. (delete {MODEL_DIR} to retrain.)')
        print('─' * 72)
    else:
        # --- Fresh path: actually train ---
        print(f'  No cached model — training YOLO26s on {DATASET_NAME}...')
        print('  (first run on a dataset: 1-15 min depending on size; later runs are cached)')
        t0 = time.time()
        try:
            from ultralytics import YOLO
            import torch
            device = pick_device()
            gpu_name = torch.cuda.get_device_name(0) if device == '0' else 'cpu'
            print(f'  device: {device} ({gpu_name})')
            print(f'  data.yaml: {YAML_P}')

            _epochs = 8 if _n_train < 200 else 30
            print(f'  epochs: {_epochs}  (auto-sized for {_n_train} train images)')

            model = YOLO('yolo26s.pt')
            # patience=3 (not the Ultralytics default of 15): for short runs
            # like our 8-epoch demo on recycling_demo, patience=15 never fires
            # because the epoch cutoff happens first. patience=3 gives early
            # stopping teeth on small data while still being forgiving on
            # 30-epoch runs on recycling_v3.
            model.train(
                data=str(YAML_P),
                epochs=_epochs,
                imgsz=640,
                batch=16,
                device=device,
                project=str(MODEL_DIR.parent),
                name=DATASET_NAME,
                exist_ok=True,
                patience=3,
                verbose=True,
                plots=False,
            )
            train_time = time.time() - t0
            state.metric(f'train_time_{DATASET_NAME}', round(train_time, 1))
            print(f'\n  Training complete in {train_time/60:.1f} min')
            if BEST_PT.exists():
                print(f'    best.pt: {BEST_PT} ({BEST_PT.stat().st_size / 1e6:.1f} MB)')
                state.active_model_path = str(BEST_PT)
                state.active_dataset = DATASET_NAME
                try:
                    with open(RESULTS_CSV) as _f:
                        _last = list(csv.DictReader(_f))[-1]
                    _map50 = float(_last.get('metrics/mAP50(B)', '0').strip())
                    state.metric(f'map50_{DATASET_NAME}', _map50)
                    print(f'    mAP50: {_map50:.3f}')
                    state.log(
                        'cell-8',
                        action='trained',
                        dataset=DATASET_NAME,
                        mAP50=_map50,
                        train_time=round(train_time, 1),
                    )
                except Exception as _e:
                    print(f'    (could not read final mAP from results.csv: {_e})')
                    state.log(
                        'cell-8',
                        action='trained',
                        dataset=DATASET_NAME,
                        train_time=round(train_time, 1),
                    )
        except Exception as _e:
            print(f'  ✗ Training failed: {type(_e).__name__}: {_e}')
            state.log(
                'cell-8',
                action='failed',
                reason='training-error',
                error=str(_e),
                dataset=DATASET_NAME,
            )
        print()
        print('─' * 72)
        print('  Trained. To re-run for cached output, re-run this cell.')
        print(f'  To force retrain:  !rm -rf {MODEL_DIR}  then re-run this cell.')
        print('─' * 72)
    print('  Next: cell 9 (compare).')
